# Reads to Results: airway DESeq2 analysis

This notebook reproduces lessons 8 to 10 of the course: count matrix → DESeq2 → plots → GO enrichment, on the `airway` dataset.

**Runtime:** this is an R notebook. In Google Colab choose *Runtime → Change runtime type → R* before running.

The first cell installs Bioconductor packages from source and takes about 10 minutes. The analysis itself takes under a minute.

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("airway", "DESeq2", "apeglm", "org.Hs.eg.db", "clusterProfiler", "pheatmap"),
                     update = FALSE, ask = FALSE)

## Lesson 8: DESeq2

In [ ]:
suppressPackageStartupMessages({ library(airway); library(DESeq2); library(apeglm); library(org.Hs.eg.db) })
data(airway)
se <- airway
se$dex <- relevel(se$dex, "untrt")
colData(se)[, c("cell", "dex")]

In [ ]:
dds <- DESeqDataSet(se, design = ~ cell + dex)
keep <- rowSums(counts(dds) >= 10) >= 4
dds <- dds[keep, ]
cat("genes after filter:", nrow(dds), "\n")   # expected: 16139
dds <- DESeq(dds)
res <- results(dds, contrast = c("dex", "trt", "untrt"), alpha = 0.05)
summary(res)                                    # expected: 2190 up, 1891 down at padj < 0.05

In [ ]:
res$symbol <- mapIds(org.Hs.eg.db, rownames(res), "SYMBOL", "ENSEMBL")
resLFC <- lfcShrink(dds, coef = "dex_trt_vs_untrt", type = "apeglm")
head(res[order(res$padj), ], 10)
res["ENSG00000103196", ]   # CRISPLD2

## Lesson 9: plots

In [ ]:
vsd <- vst(dds, blind = FALSE)
plotPCA(vsd, intgroup = c("dex", "cell"))
plotMA(res, ylim = c(-5, 5), main = "unshrunken")
plotMA(resLFC, ylim = c(-5, 5), main = "apeglm")
plotDispEsts(dds)
plotCounts(dds, gene = "ENSG00000103196", intgroup = c("dex", "cell"))
hist(res$pvalue[res$baseMean > 1], breaks = 20, col = "grey70", main = "p-values", xlab = "p")

In [ ]:
library(pheatmap)
topVar <- head(order(rowVars(assay(vsd)), decreasing = TRUE), 25)
mat <- assay(vsd)[topVar, ]
rownames(mat) <- ifelse(is.na(res$symbol[topVar]), rownames(mat), res$symbol[topVar])
ann <- as.data.frame(colData(vsd)[, c("dex", "cell")])
pheatmap(mat, scale = "row", annotation_col = ann)

## Lesson 10: GO enrichment

In [ ]:
library(clusterProfiler)
sig  <- rownames(res)[which(res$padj < 0.05 & abs(res$log2FoldChange) > 1)]
univ <- rownames(res)[!is.na(res$padj)]
cat(length(sig), "genes of", length(univ), "\n")   # expected: 951 of 15826
ego <- enrichGO(gene = sig, universe = univ, OrgDb = org.Hs.eg.db, keyType = "ENSEMBL",
                ont = "BP", pAdjustMethod = "BH", qvalueCutoff = 0.05, readable = TRUE)
head(as.data.frame(ego)[, c("ID", "Description", "GeneRatio", "p.adjust")], 15)
dotplot(ego, showCategory = 15)

In [ ]:
write.csv(as.data.frame(res[order(res$padj), ]), "airway_deseq2_results.csv")
sessionInfo()